In [50]:
import pandas as pd

In [51]:
# lattitude
# longitude
# ncount
# last mile leg 1 counts
# last mile leg 2 counts
# combined mode share
# avg last mile cost and time
# avg locatlity distance
# Dominant leg 1 mode
# Dominant leg 2 mode
# Dominant last mile mode


In [134]:

blr = pd.read_csv(r'E:\aug-2025\Introduction-mms\project-2\cityData\cleaned_survey_data_nagpur.csv')
location = pd.read_csv(r'E:\aug-2025\Introduction-mms\project-2\cityData\nagpur_metro_v2.csv')
location.columns

Index(['full_id', 'osm_id', 'osm_type', 'railway', 'name', 'Latitude',
       'Longitude'],
      dtype='object')

In [135]:
# Remove null values from location
location=location[['name', 'Latitude', 'Longitude']]


In [136]:
df = pd.merge(blr, location, how='left', left_on='Which is the **Origin** metro station of this trip? / Which is the **Destination** metro station of this trip?', right_on='name')
print(df.columns)

Index(['Respondent ID', 'Metro Station at which the survey is being conducted',
       'Location within station where survey is being conducted (example – entry gate/exit gate/platform).',
       'Which is the **Origin** metro station of this trip? / Which is the **Destination** metro station of this trip?',
       'Frequency of the trip', 'purpose of this trip',
       'When do you make the return ', 'First mile connection (Leg 1)',
       'First mile connection (Leg 2)', 'Primary Mode (Leg 1)',
       'Primary Mode (Leg 2)', 'Last Mile connection (Leg 1)',
       'Last Mile connection (Leg 2)',
       'F1Time: How many minutes is your trip by {INSERTANS:336587X78X8321} ?',
       'F2Cost: How much do you pay for the trip by {INSERTANS:336587X78X8321} ?  [Not applicable for walk trips. Insert 0 for personal bicycle]',
       'F3: How far do you travel in terms of distance by  {INSERTANS:336587X78X8321}?(Insert 0 if not known)  Distance (in KM):',
       'F4Time: How many minutes is yo

In [137]:
data = df[['name', 'Latitude_y', 'Longitude_y','Last Mile connection (Leg 1)', 'Last Mile connection (Leg 2)','Last Mile Mode Sharing']]

In [139]:
null_long = data['name'].isnull().sum()
print(null_long)
null_lat = data['Latitude_y'].isnull().sum()
print(null_lat)
print(len(location))
print(len(data['name'].unique()))

0
0
37
23


In [ ]:
# # In data we have 50 location but in location we have 63 locations. I want the remaining 13 locations also in the data
# d = location.copy()
# d = d[~d['Station'].isin(data['Station_y'])]
# data = pd.concat([data, d], ignore_index=True)
# d = d.reset_index(drop=True)
# print(len(data['Station_y'].unique()))

196


In [141]:


# Drop rows with null values
data = data.dropna()

# column wise null values
data.isnull().sum()

name                            0
Latitude_y                      0
Longitude_y                     0
Last Mile connection (Leg 1)    0
Last Mile connection (Leg 2)    0
Last Mile Mode Sharing          0
dtype: int64

In [143]:
len(data.columns.to_list())
data.columns.to_list()

['name',
 'Latitude_y',
 'Longitude_y',
 'Last Mile connection (Leg 1)',
 'Last Mile connection (Leg 2)',
 'Last Mile Mode Sharing']

In [144]:
# Rename columns for better understanding
data = data.rename(columns={
    'name': 'Metro Station',
    'Latitude_y': 'Latitude',
    'Longitude_y': 'Longitude',
    'Last Mile connection (Leg 1)': 'Last Mile Connection Leg 1',
    'Last Mile connection (Leg 2)': 'Last Mile Connection Leg 2',
    'Last Mile Mode Sharing': 'Last Mile Mode Sharing'
})

In [ ]:
# # Make a copy of data
# data_final = data.copy()

# # Tosave the final dataframe to a CSV file
# data_final.to_csv(r'E:\aug-2025\Introduction-mms\project-2\cityData\metro_blr_data.csv', index=False)

In [145]:


print(data.columns.tolist())

data.head(2)

['Metro Station', 'Latitude', 'Longitude', 'Last Mile Connection Leg 1', 'Last Mile Connection Leg 2', 'Last Mile Mode Sharing']


,Metro Station,Latitude,Longitude,Last Mile Connection Leg 1,Last Mile Connection Leg 2,Last Mile Mode Sharing
0,Sitabuldi (NS and EW),21.14145,79.08248,Walk,No Last mile leg 2,Walk + No Last mile leg 2
1,Ujjwal Nagar,21.09635,79.06632,Two-wheeler,No Last mile leg 2,Two-wheeler + No Last mile leg 2


In [ ]:
# base_summary = data.groupby(
#     ["Metro Station", "Latitude", "Longitude"], as_index=False
# ).agg(
#     trips=("Metro Station", "count"),
#     avg_time_leg1=("Trip Time Leg 1 (minutes)", "mean"),
#     avg_time_leg2=("Trip Time Leg 2 (minutes)", "mean"),
#     total_avg_time=("Last Mile Mode Share Trip Time", "mean"),
#     avg_cost_leg1=("Trip Cost Leg 1 (Rupees)", "mean"),
#     avg_cost_leg2=("Trip Cost Leg 2 (Rupees)", "mean"),
#     total_avg_cost=("Last Mile Mode Share Cost", "mean"),
#     avg_dist_leg1=("Total Last Mile Leg 1 Distance", "mean"),
#     avg_dist_leg2=("Total Last Mile Leg 2 Distance", "mean"),
#     total_avg_dist=("Total Last Mile Mode Share Distance", "mean")
# )

In [146]:
# --- Step 2: Explode multiple columns ---
def explode_counts(df, column, prefix):
    """
    Splits comma-separated values in `column`,
    explodes them into rows, and counts per station.
    """
    df_exp = df.assign(split=df[column].str.split(",\s*")).explode("split")
    return (
        pd.crosstab(
            [df_exp["Metro Station"], df_exp["Latitude"], df_exp["Longitude"]],
            df_exp["split"]
        )
        .add_prefix(prefix + "_")
    )

# Apply for each of your three columns
leg1_counts = explode_counts(data, "Last Mile Connection Leg 1", "Leg1")
leg2_counts = explode_counts(data, "Last Mile Connection Leg 2", "Leg2")
share_counts = explode_counts(data, "Last Mile Mode Sharing", "Share")

# --- Step 3: Merge everything ---
summary = leg1_counts.reset_index()
summary = summary.merge(leg2_counts, on=["Metro Station", "Latitude", "Longitude"], how="left")
summary = summary.merge(share_counts, on=["Metro Station", "Latitude", "Longitude"], how="left")

# Replace NaN with 0 for counts
summary = summary.fillna(0)


<>:7: SyntaxWarning: invalid escape sequence '\s'
<>:7: SyntaxWarning: invalid escape sequence '\s'
C:\Users\maddi\AppData\Local\Temp\ipykernel_21468\2674831239.py:7: SyntaxWarning: invalid escape sequence '\s'
  df_exp = df.assign(split=df[column].str.split(",\s*")).explode("split")


In [147]:
# save the summary to a CSV file
summary.to_csv(r'E:\aug-2025\Introduction-mms\project-2\cityData\metro_nagpur_summary.csv', index=False)